In [9]:
import pygame
import numpy as np
import random
import math
import time
import sys

# Initialize pygame
pygame.init()

# Constants
WIDTH, HEIGHT = 600, 600
BOARD_SIZE = 8
CELL_SIZE = WIDTH // BOARD_SIZE
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
HIGHLIGHT = (255, 200, 0)  # Highlight color for moved queens
TEXT_COLOR = (50, 50, 50)
BUTTON_COLOR = (100, 150, 200)
BUTTON_HOVER = (120, 170, 220)
BUTTON_TEXT = (255, 255, 255)

# Try to load Queen Image, use fallback if file not found
try:
    QUEEN_IMAGE = pygame.image.load("queen.png")
    QUEEN_IMAGE = pygame.transform.scale(QUEEN_IMAGE, (CELL_SIZE - 10, CELL_SIZE - 10))
except:
    # Fallback - no image
    QUEEN_IMAGE = None

# Initialize Pygame window
screen = pygame.display.set_mode((WIDTH, HEIGHT + 150))  # Extra height for UI
pygame.display.set_caption("8-Queens Problem - Local Search")

# Initialize font
pygame.font.init()
font = pygame.font.SysFont('Arial', 16)
large_font = pygame.font.SysFont('Arial', 20, bold=True)

# Function to calculate conflicts (heuristic)
def count_conflicts(board):
    conflicts = 0
    for i in range(BOARD_SIZE):
        for j in range(i + 1, BOARD_SIZE):
            # Check if queens attack each other
            if board[i] == board[j]:  # Same column
                conflicts += 1
            elif abs(board[i] - board[j]) == abs(i - j):  # Diagonal
                conflicts += 1
    return conflicts

# Function to generate all neighbors (move one queen to any position in its row)
def get_neighbors(board):
    neighbors = []
    for row in range(BOARD_SIZE):
        for col in range(BOARD_SIZE):
            if col != board[row]:
                new_board = board.copy()
                new_board[row] = col
                neighbors.append((new_board, row))  # Include which row was changed
    return neighbors

# Function to draw the board
def draw_board(board, last_moved_row=None):
    screen.fill(WHITE)
    
    # Draw board
    for row in range(BOARD_SIZE):
        for col in range(BOARD_SIZE):
            # Draw cell
            color = BLACK if (row + col) % 2 == 0 else WHITE
            pygame.draw.rect(screen, color, (col * CELL_SIZE, row * CELL_SIZE, CELL_SIZE, CELL_SIZE))
            
            # Draw queen
            if board[row] == col:
                # Highlight the last moved queen
                if row == last_moved_row:
                    pygame.draw.rect(screen, HIGHLIGHT, 
                                    (col * CELL_SIZE + 2, row * CELL_SIZE + 2, 
                                     CELL_SIZE - 4, CELL_SIZE - 4))
                
                if QUEEN_IMAGE:
                    screen.blit(QUEEN_IMAGE, (col * CELL_SIZE + 5, row * CELL_SIZE + 5))
                else:
                    # Draw a circle as fallback
                    pygame.draw.circle(screen, (200, 0, 0), 
                                      (col * CELL_SIZE + CELL_SIZE//2, 
                                       row * CELL_SIZE + CELL_SIZE//2), 
                                       CELL_SIZE//3)
    
    # Draw conflicts count
    conflicts = count_conflicts(board)
    text = large_font.render(f"Conflicts: {conflicts}", True, TEXT_COLOR)
    screen.blit(text, (20, HEIGHT + 20))
    
    pygame.display.update()

# Function to draw UI
def draw_ui(algorithm_name, iteration, temperature=None, k=None):
    # Clear UI area
    pygame.draw.rect(screen, WHITE, (0, HEIGHT, WIDTH, 150))
    
    # Draw algorithm name
    text = large_font.render(f"Algorithm: {algorithm_name}", True, TEXT_COLOR)
    screen.blit(text, (WIDTH//2 - text.get_width()//2, HEIGHT + 60))
    
    # Draw iteration counter
    text = font.render(f"Iteration: {iteration}", True, TEXT_COLOR)
    screen.blit(text, (20, HEIGHT + 100))
    
    # Draw algorithm-specific info
    if algorithm_name == "Simulated Annealing" and temperature is not None:
        text = font.render(f"Temperature: {temperature:.2f}", True, TEXT_COLOR)
        screen.blit(text, (WIDTH//2, HEIGHT + 100))
    elif algorithm_name == "Local Beam Search" and k is not None:
        text = font.render(f"Beam Width: {k}", True, TEXT_COLOR)
        screen.blit(text, (WIDTH//2, HEIGHT + 100))
    
    pygame.display.update()

# Generate initial board - each row has one queen in a random column
def generate_initial_board():
    return np.random.randint(0, BOARD_SIZE, BOARD_SIZE)

# Hill Climbing
def hill_climbing():
    # Setup
    board = generate_initial_board()
    iteration = 0
    current_conflicts = count_conflicts(board)
    
    draw_board(board)
    draw_ui("Hill Climbing", iteration)
    
    # Main loop
    while current_conflicts > 0:
        iteration += 1
        
        # Get all neighbors
        neighbors = get_neighbors(board)
        
        # Find the best neighbor
        best_neighbor = None
        best_conflicts = current_conflicts
        best_row_moved = None
        
        for neighbor, row_moved in neighbors:
            neighbor_conflicts = count_conflicts(neighbor)
            if neighbor_conflicts < best_conflicts:
                best_neighbor = neighbor
                best_conflicts = neighbor_conflicts
                best_row_moved = row_moved
        
        # If no better neighbor, we've reached a local optimum
        if best_conflicts >= current_conflicts:
            print(f"Stuck at local optimum with {current_conflicts} conflicts after {iteration} iterations")
            return board, iteration, current_conflicts
        
        # Move to the best neighbor
        board = best_neighbor
        current_conflicts = best_conflicts
        
        # Visualization
        draw_board(board, best_row_moved)
        draw_ui("Hill Climbing", iteration)
        
        # Handle events
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()
        
        pygame.time.delay(200)  # Slow down to show steps
    
    print(f"Solution found in {iteration} iterations")
    return board, iteration, 0

# Simulated Annealing
def simulated_annealing():
    # Parameters
    initial_temp = 10.0  # Starting temperature
    cooling_rate = 0.95  # How quickly temperature decreases
    
    # Setup
    board = generate_initial_board()
    iteration = 0
    current_conflicts = count_conflicts(board)
    temp = initial_temp
    
    draw_board(board)
    draw_ui("Simulated Annealing", iteration, temp)
    
    # Main loop
    while current_conflicts > 0 and temp > 0.01:
        iteration += 1
        
        # Randomly select a neighbor (move one queen in a random row)
        row = random.randint(0, BOARD_SIZE - 1)
        new_col = random.randint(0, BOARD_SIZE - 1)
        while new_col == board[row]:  # Ensure we're actually moving
            new_col = random.randint(0, BOARD_SIZE - 1)
        
        # Create new neighbor
        new_board = board.copy()
        new_board[row] = new_col
        new_conflicts = count_conflicts(new_board)
        
        # Decide whether to accept the move
        delta_E = new_conflicts - current_conflicts
        
        if delta_E < 0:  # Better solution, always accept
            board = new_board
            current_conflicts = new_conflicts
            row_moved = row
        else:  # Worse solution, accept with probability
            probability = math.exp(-delta_E / temp)
            if random.random() < probability:
                board = new_board
                current_conflicts = new_conflicts
                row_moved = row
            else:
                row_moved = None
        
        # Cooling schedule
        temp *= cooling_rate
        
        # Visualization
        draw_board(board, row_moved)
        draw_ui("Simulated Annealing", iteration, temp)
        
        # Handle events
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()
        
        pygame.time.delay(100)  # Slow down to show steps
    
    print(f"Solution found in {iteration} iterations with {current_conflicts} conflicts")
    return board, iteration, current_conflicts

# Local Beam Search
def local_beam_search(k=5):
    # Generate k random starting states
    states = [generate_initial_board() for _ in range(k)]
    iteration = 0
    
    # Calculate conflicts for all states
    conflicts = [count_conflicts(state) for state in states]
    
    # Find the best state so far
    best_index = conflicts.index(min(conflicts))
    
    draw_board(states[best_index])
    draw_ui("Local Beam Search", iteration, k=k)
    
    # Main loop
    while min(conflicts) > 0:
        iteration += 1
        all_neighbors = []
        
        # Generate all neighbors for all states
        for state in states:
            neighbors = [(neighbor, row) for neighbor, row in get_neighbors(state)]
            all_neighbors.extend(neighbors)
        
        # Calculate conflicts for all neighbors
        neighbor_conflicts = [(count_conflicts(neighbor), i, row) for i, (neighbor, row) in enumerate(all_neighbors)]
        
        # Sort by conflicts (ascending)
        neighbor_conflicts.sort()
        
        # Select k best neighbors
        if len(neighbor_conflicts) <= k:
            best_neighbors = neighbor_conflicts
        else:
            best_neighbors = neighbor_conflicts[:k]
        
        # Update states
        new_states = []
        for _, i, _ in best_neighbors:
            new_states.append(all_neighbors[i][0])
        
        states = new_states
        conflicts = [count_conflicts(state) for state in states]
        
        # Find best state for visualization
        best_index = conflicts.index(min(conflicts))
        best_state = states[best_index]
        
        # For visualization purposes, try to find which row changed in the best state
        row_moved = all_neighbors[best_neighbors[0][1]][1] if best_neighbors else None
        
        # Visualization
        draw_board(best_state, row_moved)
        draw_ui("Local Beam Search", iteration, k=k)
        
        # Handle events
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()
        
        pygame.time.delay(100)  # Slow down to show steps
    
    best_index = conflicts.index(min(conflicts))
    best_state = states[best_index]
    best_conflicts = conflicts[best_index]
    
    print(f"Solution found in {iteration} iterations with {best_conflicts} conflicts")
    return best_state, iteration, best_conflicts

# Button class for UI
class Button:
    def __init__(self, x, y, width, height, text, callback):
        self.rect = pygame.Rect(x, y, width, height)
        self.text = text
        self.callback = callback
        self.hovered = False
    
    def draw(self):
        color = BUTTON_HOVER if self.hovered else BUTTON_COLOR
        pygame.draw.rect(screen, color, self.rect, border_radius=5)
        
        text_surf = font.render(self.text, True, BUTTON_TEXT)
        text_rect = text_surf.get_rect(center=self.rect.center)
        screen.blit(text_surf, text_rect)
    
    def update(self, mouse_pos):
        self.hovered = self.rect.collidepoint(mouse_pos)
    
    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.rect.collidepoint(event.pos):
                self.callback()
                return True
        return False

# Results display
def display_results(results):
    screen.fill(WHITE)
    
    # Display header
    header = large_font.render("Algorithm Comparison Results", True, TEXT_COLOR)
    screen.blit(header, (WIDTH//2 - header.get_width()//2, 20))
    
    # Display results table
    y = 80
    headers = ["Algorithm", "Iterations", "Conflicts"]
    
    # Table headers
    for i, header_text in enumerate(headers):
        text = font.render(header_text, True, TEXT_COLOR)
        screen.blit(text, (50 + i * 180, y))
    
    y += 30
    
    # Table data
    for algorithm, (_, iterations, conflicts) in results.items():
        pygame.draw.line(screen, BLACK, (40, y-5), (WIDTH-40, y-5), 1)
        
        # Algorithm name
        text = font.render(algorithm, True, TEXT_COLOR)
        screen.blit(text, (50, y))
        
        # Iterations
        text = font.render(str(iterations), True, TEXT_COLOR)
        screen.blit(text, (50 + 180, y))
        
        # Conflicts
        text = font.render(str(conflicts), True, TEXT_COLOR)
        screen.blit(text, (50 + 360, y))
        
        y += 30
    
    # Add restart button
    restart_button = Button(WIDTH//2 - 75, HEIGHT - 60, 150, 40, "Run Again", main)
    restart_button.draw()
    
    pygame.display.update()
    
    # Wait for user to quit or restart
    waiting = True
    while waiting:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()
            elif event.type == pygame.MOUSEMOTION:
                restart_button.update(event.pos)
                restart_button.draw()
                pygame.display.update()
            elif restart_button.handle_event(event):
                waiting = False
        
        pygame.time.delay(50)

# Main function
def main():
    screen.fill(WHITE)
    
    # Create buttons
    button_width = 180
    button_height = 50
    buttons = [
        Button(WIDTH//2 - button_width//2, 150, button_width, button_height, 
               "Hill Climbing", lambda: run_algorithm("Hill Climbing")),
        Button(WIDTH//2 - button_width//2, 220, button_width, button_height, 
               "Simulated Annealing", lambda: run_algorithm("Simulated Annealing")),
        Button(WIDTH//2 - button_width//2, 290, button_width, button_height, 
               "Local Beam Search", lambda: run_algorithm("Local Beam Search")),
        Button(WIDTH//2 - button_width//2, 360, button_width, button_height, 
               "Compare All", lambda: run_algorithm("Compare All"))
    ]
    
    # Title
    title = large_font.render("8-Queens Problem - Local Search Algorithms", True, TEXT_COLOR)
    screen.blit(title, (WIDTH//2 - title.get_width()//2, 80))
    
    # Draw buttons
    for button in buttons:
        button.draw()
    
    pygame.display.update()
    
    # Event loop
    running = True
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
                pygame.quit()
                sys.exit()
            elif event.type == pygame.MOUSEMOTION:
                for button in buttons:
                    button.update(event.pos)
                    button.draw()
                pygame.display.update()
            elif event.type == pygame.MOUSEBUTTONDOWN:
                for button in buttons:
                    if button.handle_event(event):
                        running = False
        
        pygame.time.delay(50)

def run_algorithm(algorithm_name):
    # Dictionary to store results
    results = {}
    
    if algorithm_name == "Hill Climbing" or algorithm_name == "Compare All":
        # Run Hill Climbing
        board, iterations, conflicts = hill_climbing()
        results["Hill Climbing"] = (board, iterations, conflicts)
    
    if algorithm_name == "Simulated Annealing" or algorithm_name == "Compare All":
        # Run Simulated Annealing
        board, iterations, conflicts = simulated_annealing()
        results["Simulated Annealing"] = (board, iterations, conflicts)
    
    if algorithm_name == "Local Beam Search" or algorithm_name == "Compare All":
        # Run Local Beam Search
        board, iterations, conflicts = local_beam_search(k=5)
        results["Local Beam Search"] = (board, iterations, conflicts)
    
    # Display comparison results
    if algorithm_name == "Compare All":
        display_results(results)
    else:
        # Display the final state for a while then return to menu
        pygame.time.delay(3000)
        main()

if __name__ == "__main__":
    main()

Solution found in 2 iterations
Solution found in 135 iterations with 1 conflicts


SystemExit: 